# Milestone 2 — Data Cleaning and Transformation

## Dataset APBD Pemerintah Daerah Indonesia

### Tujuan
Notebook ini digunakan untuk membersihkan dan mentransformasi
dataset APBD Pemerintah Daerah Indonesia agar siap digunakan
untuk analisis lanjutan.

### Tools
- Polars: data processing dan cleaning
- DuckDB: validasi data menggunakan SQL

### Prinsip Pengolahan
- Data mentah tidak diubah.
- Proses cleaning dapat direproduksi.
- Perubahan data harus berdasarkan temuan.
- Hasil cleaning divalidasi sebelum disimpan.

In [34]:
import polars as pl
import duckdb
from pathlib import Path

In [35]:
DATA_PATH = Path("../data/raw/apbd_pemda_indonesia.csv")
OUTPUT_DIR = Path("../data/processed/")

print("File input tersedia:", DATA_PATH.exists())
print("Lokasi output:", OUTPUT_DIR.resolve())

File input tersedia: True
Lokasi output: D:\SEMESTER 7\Analisis Big Data\tugas1-abd-2026\data\processed


In [36]:
df_raw = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10000,
    try_parse_dates=False
)

schema_raw = df_raw.collect_schema()

print("Jumlah kolom:", len(schema_raw))
print("\nStruktur kolom:")
print(schema_raw)

Jumlah kolom: 11

Struktur kolom:
Schema({'tahun': Int64, 'bulan': Int64, 'id_provinsi': Int64, 'provinsi': String, 'id_pemda': Int64, 'pemda': String, 'no_akun': Int64, 'akun': String, 'anggaran': Float64, 'realisasi': Float64, 'persentase': Float64})


In [37]:
row_count = df_raw.select(pl.len().alias("jumlah_baris")).collect()

print("Jumlah baris:", row_count["jumlah_baris"][0])

df_raw.head(5).collect()

Jumlah baris: 1155181


tahun,bulan,id_provinsi,provinsi,id_pemda,pemda,no_akun,akun,anggaran,realisasi,persentase
i64,i64,i64,str,i64,str,i64,str,f64,f64,f64
2011,12,1,"""Provinsi Aceh""",0,"""Prov. Aceh""",1,"""Pendapatan""",7.0894e12,7.6093e12,107.33
2011,12,1,"""Provinsi Aceh""",0,"""Prov. Aceh""",2,"""PAD""",7.9728e11,8.0284e11,100.7
2011,12,1,"""Provinsi Aceh""",0,"""Prov. Aceh""",3,"""Pajak Daerah""",6.2271e11,5.8618e11,94.13
2011,12,1,"""Provinsi Aceh""",0,"""Prov. Aceh""",4,"""Retribusi Daerah""",1.3264e10,9.4160e9,70.99
2011,12,1,"""Provinsi Aceh""",0,"""Prov. Aceh""",5,"""Hasil Pengelolaan Kekayaan Dae…",1.0200e11,6.6007e10,64.71


In [38]:
null_counts = df_raw.select([
    pl.col(col).null_count().alias(col)
    for col in schema_raw.names()
]).collect()

null_counts

tahun,bulan,id_provinsi,provinsi,id_pemda,pemda,no_akun,akun,anggaran,realisasi,persentase
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0


In [39]:
total_rows = df_raw.select(pl.len()).collect().item()

unique_rows = (
    df_raw
    .unique()
    .select(pl.len())
    .collect()
    .item()
)

print("Total baris :", total_rows)
print("Baris unik  :", unique_rows)
print("Duplikat    :", total_rows - unique_rows)

Total baris : 1155181
Baris unik  : 905781
Duplikat    : 249400


In [40]:
invalid_month = (
    df_raw
    .filter(
        (pl.col("bulan") < 1) |
        (pl.col("bulan") > 12)
    )
    .select(pl.len())
    .collect()
    .item()
)

print("Jumlah bulan tidak valid:", invalid_month)

Jumlah bulan tidak valid: 0


In [41]:
invalid_year = (
    df_raw
    .filter(
        (pl.col("tahun") < 2011) |
        (pl.col("tahun") > 2025)
    )
    .select(pl.len())
    .collect()
    .item()
)

print("Jumlah tahun tidak valid:", invalid_year)

Jumlah tahun tidak valid: 140103


In [42]:
year_distribution = (
    df_raw
    .group_by("tahun")
    .agg(pl.len().alias("jumlah_baris"))
    .sort("tahun")
    .collect()
)

year_distribution

tahun,jumlah_baris
i64,u32
2011,33140
2012,32894
2013,32942
2014,33502
2015,33906
…,…
2022,31344
2023,196680
2024,209241


In [43]:
negative_values = (
    df_raw
    .filter(
        (pl.col("anggaran") < 0) |
        (pl.col("realisasi") < 0)
    )
    .select(pl.len())
    .collect()
    .item()
)

print("Jumlah baris dengan anggaran/realisasi negatif:", negative_values)

Jumlah baris dengan anggaran/realisasi negatif: 19222


In [44]:
negative_summary = df_raw.select([
    (pl.col("anggaran") < 0).sum().alias("anggaran_negatif"),
    (pl.col("realisasi") < 0).sum().alias("realisasi_negatif"),
    (
        (pl.col("anggaran") < 0) &
        (pl.col("realisasi") < 0)
    ).sum().alias("keduanya_negatif")
]).collect()

negative_summary

anggaran_negatif,realisasi_negatif,keduanya_negatif
u32,u32,u32
15211,14746,10735


In [45]:
negative_examples = (
    df_raw
    .filter(
        (pl.col("anggaran") < 0) |
        (pl.col("realisasi") < 0)
    )
    .select([
        "tahun",
        "bulan",
        "provinsi",
        "pemda",
        "no_akun",
        "akun",
        "anggaran",
        "realisasi",
        "persentase"
    ])
    .head(15)
    .collect()
)

negative_examples

tahun,bulan,provinsi,pemda,no_akun,akun,anggaran,realisasi,persentase
i64,i64,str,str,i64,str,f64,f64,f64
2011,12,"""Provinsi Aceh""","""Prov. Aceh""",29,"""Pengeluaran Pembiayaan""",0.0,-3.3580e10,0.0
2011,12,"""Provinsi Aceh""","""Prov. Aceh""",30,"""Pengeluaran Perhitungan Pihak …",0.0,-3.3580e10,0.0
2011,12,"""Provinsi Aceh""","""Kab. Aceh Barat""",30,"""Pengeluaran Pembiayaan""",-1.8500e10,-1.8500e10,100.0
2011,12,"""Provinsi Aceh""","""Kab. Aceh Barat""",31,"""Pembayaran Pokok Utang""",-1.8500e10,-1.8500e10,100.0
2011,12,"""Provinsi Aceh""","""Kab. Aceh Besar""",33,"""Pengeluaran Pembiayaan""",0.0,-1.0000e9,0.0
…,…,…,…,…,…,…,…,…
2011,12,"""Provinsi Aceh""","""Kab. Aceh Singkil""",31,"""Pengeluaran Pembiayaan""",-4.0000e8,0.0,0.0
2011,12,"""Provinsi Aceh""","""Kab. Aceh Singkil""",32,"""Penyertaan Modal""",-4.0000e8,0.0,0.0
2011,12,"""Provinsi Aceh""","""Kab. Aceh Tengah""",31,"""Pengeluaran Pembiayaan""",-3.5470e9,-3.8315e9,108.02


In [46]:
negative_by_account = (
    df_raw
    .filter(
        (pl.col("anggaran") < 0) |
        (pl.col("realisasi") < 0)
    )
    .group_by("akun")
    .agg([
        pl.len().alias("jumlah_baris"),
        (pl.col("anggaran") < 0).sum().alias("anggaran_negatif"),
        (pl.col("realisasi") < 0).sum().alias("realisasi_negatif"),
    ])
    .sort("jumlah_baris", descending=True)
    .collect()
)

negative_by_account

akun,jumlah_baris,anggaran_negatif,realisasi_negatif
str,u32,u32,u32
"""Pembiayaan Daerah""",4872,3799,1968
"""Pengeluaran Pembiayaan""",4664,4328,4402
"""Penyertaan Modal""",4232,3812,3720
"""Pembayaran Pokok Utang""",2458,1912,2186
"""Pemberian Pinjaman Daerah""",758,594,564
…,…,…,…
"""Belanja Subsidi""",2,0,2
"""Belanja Bunga""",1,0,1
"""Penerimaan Pinjaman Daerah""",1,0,1


In [47]:
percentage_summary = df_raw.select([
    pl.col("persentase").min().alias("persentase_min"),
    pl.col("persentase").max().alias("persentase_max"),
    (pl.col("persentase") < 0).sum().alias("persentase_negatif"),
    (pl.col("persentase") > 100).sum().alias("persentase_di_atas_100"),
    pl.col("persentase").is_nan().sum().alias("persentase_nan"),
]).collect()

percentage_summary

persentase_min,persentase_max,persentase_negatif,persentase_di_atas_100,persentase_nan
f64,f64,u32,u32,u32
-997.48,999.98,2808,220697,0


In [48]:
extreme_percentages = (
    df_raw
    .filter(
        (pl.col("persentase") < 0) |
        (pl.col("persentase") > 100)
    )
    .select([
        "tahun",
        "bulan",
        "provinsi",
        "pemda",
        "no_akun",
        "akun",
        "anggaran",
        "realisasi",
        "persentase"
    ])
    .sort("persentase")
    .head(10)
    .collect()
)

extreme_percentages

tahun,bulan,provinsi,pemda,no_akun,akun,anggaran,realisasi,persentase
i64,i64,str,str,i64,str,f64,f64,f64
2021,12,"""Provinsi Nusa Tenggara Timur""","""Kab. Alor""",29,"""Pembiayaan Daerah""",-3.0000e9,2.9924e10,-997.48
2021,12,"""Provinsi Nusa Tenggara Timur""","""Kab. Alor""",29,"""Pembiayaan Daerah""",-3.0000e9,2.9924e10,-997.48
2020,12,"""Provinsi Kalimantan Tengah""","""Kab. Gunung Mas""",30,"""Pembiayaan Daerah""",-2.5914e9,2.5813e10,-996.09
2020,12,"""Provinsi Kalimantan Tengah""","""Kab. Gunung Mas""",30,"""Pembiayaan Daerah""",-2.5914e9,2.5813e10,-996.09
2019,12,"""Provinsi Sulawesi Tengah""","""Kab. Poso""",25,"""Pembiayaan Daerah""",-4.7021e9,4.6154e10,-981.55
2019,12,"""Provinsi Sulawesi Tengah""","""Kab. Poso""",25,"""Pembiayaan Daerah""",-4.7021e9,4.6154e10,-981.55
2023,6,"""Provinsi Jawa Barat""","""Provinsi Jawa Barat""",27,"""Pembiayaan Daerah""",-2.1446e11,2.0981e12,-978.31
2025,5,"""Provinsi Nusa Tenggara Timur""","""Kota Kupang""",23,"""Pembiayaan Daerah""",-5.0000e9,4.8730e10,-974.59
2025,7,"""Provinsi Nusa Tenggara Timur""","""Kota Kupang""",23,"""Pembiayaan Daerah""",-5.0000e9,4.8730e10,-974.59


In [49]:
duplicate_keys = (
    df_raw
    .group_by([
        "tahun",
        "bulan",
        "id_pemda",
        "no_akun"
    ])
    .agg([
        pl.len().alias("jumlah_baris")
    ])
    .filter(pl.col("jumlah_baris") > 1)
    .sort("jumlah_baris", descending=True)
    .collect()
)

print("Jumlah kombinasi kunci yang muncul lebih dari sekali:",
      duplicate_keys.height)

duplicate_keys.head(10)

Jumlah kombinasi kunci yang muncul lebih dari sekali: 66001


tahun,bulan,id_pemda,no_akun,jumlah_baris
i64,i64,i64,i64,u32
2025,12,0,20,76
2023,12,0,1,76
2023,12,0,8,76
2023,12,0,14,76
2025,12,0,9,76
2025,12,0,25,76
2024,12,0,2,76
2023,12,0,3,76
2023,12,0,23,76


In [50]:
# Hitung jumlah baris unik berdasarkan seluruh kolom
df_unique = df_raw.unique()

jumlah_unik_global = df_unique.select(pl.len()).collect().item()

jumlah_total = df_raw.select(pl.len()).collect().item()

jumlah_duplikat_global = jumlah_total - jumlah_unik_global

print("Jumlah baris awal:", jumlah_total)
print("Jumlah baris unik:", jumlah_unik_global)
print("Jumlah duplikat persis:", jumlah_duplikat_global)
print(
    "Persentase duplikat:",
    round(jumlah_duplikat_global / jumlah_total * 100, 2),
    "%"
)

Jumlah baris awal: 1155181
Jumlah baris unik: 905781
Jumlah duplikat persis: 249400
Persentase duplikat: 21.59 %


In [51]:
# Hapus baris yang identik pada seluruh kolom
df_clean = df_raw.unique(maintain_order=True)

# Simpan hasil ke folder processed
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "apbd_pemda_indonesia_clean.parquet"

df_clean.sink_parquet(OUTPUT_PATH)

print("File hasil cleaning:", OUTPUT_PATH)
print("Jumlah baris awal:", jumlah_total)
print("Jumlah baris setelah cleaning:", jumlah_unik_global)
print("Jumlah baris yang dihapus:", jumlah_duplikat_global)

File hasil cleaning: ..\data\processed\apbd_pemda_indonesia_clean.parquet
Jumlah baris awal: 1155181
Jumlah baris setelah cleaning: 905781
Jumlah baris yang dihapus: 249400


In [52]:
df_check = pl.scan_parquet(OUTPUT_PATH)

print("Jumlah baris hasil:", df_check.select(pl.len()).collect().item())
print("Jumlah duplikat tersisa:",
      df_check.select(pl.len()).collect().item()
      - df_check.unique().select(pl.len()).collect().item())

Jumlah baris hasil: 905781
Jumlah duplikat tersisa: 0


In [53]:
# Audit kualitas data setelah deduplikasi
df_clean = pl.scan_parquet(OUTPUT_PATH)

# 1. Periksa nilai kosong
null_audit = df_clean.select([
    pl.col(col).null_count().alias(col)
    for col in df_clean.collect_schema().names()
]).collect()

print("=== Nilai kosong per kolom ===")
print(null_audit)

# 2. Periksa bulan yang tidak valid
invalid_month = (
    df_clean
    .filter(~pl.col("bulan").is_between(1, 12))
    .select(pl.len())
    .collect()
    .item()
)

print("\nJumlah baris dengan bulan tidak valid:", invalid_month)

# 3. Periksa nilai anggaran dan realisasi negatif
negative_audit = df_clean.select([
    (pl.col("anggaran") < 0).sum().alias("anggaran_negatif"),
    (pl.col("realisasi") < 0).sum().alias("realisasi_negatif"),
    (
        (pl.col("anggaran") < 0) |
        (pl.col("realisasi") < 0)
    ).sum().alias("baris_dengan_nilai_negatif")
]).collect()

print("\n=== Audit nilai negatif ===")
print(negative_audit)

=== Nilai kosong per kolom ===
shape: (1, 11)
┌───────┬───────┬─────────────┬──────────┬───┬──────┬──────────┬───────────┬────────────┐
│ tahun ┆ bulan ┆ id_provinsi ┆ provinsi ┆ … ┆ akun ┆ anggaran ┆ realisasi ┆ persentase │
│ ---   ┆ ---   ┆ ---         ┆ ---      ┆   ┆ ---  ┆ ---      ┆ ---       ┆ ---        │
│ u32   ┆ u32   ┆ u32         ┆ u32      ┆   ┆ u32  ┆ u32      ┆ u32       ┆ u32        │
╞═══════╪═══════╪═════════════╪══════════╪═══╪══════╪══════════╪═══════════╪════════════╡
│ 0     ┆ 0     ┆ 0           ┆ 0        ┆ … ┆ 0    ┆ 0        ┆ 0         ┆ 0          │
└───────┴───────┴─────────────┴──────────┴───┴──────┴──────────┴───────────┴────────────┘

Jumlah baris dengan bulan tidak valid: 0

=== Audit nilai negatif ===
shape: (1, 3)
┌──────────────────┬───────────────────┬────────────────────────────┐
│ anggaran_negatif ┆ realisasi_negatif ┆ baris_dengan_nilai_negatif │
│ ---              ┆ ---               ┆ ---                        │
│ u32              ┆ u32     

In [54]:
negative_by_account = (
    df_clean
    .filter(
        (pl.col("anggaran") < 0) |
        (pl.col("realisasi") < 0)
    )
    .group_by("akun")
    .agg([
        pl.len().alias("jumlah_baris"),
        (pl.col("anggaran") < 0).sum().alias("anggaran_negatif"),
        (pl.col("realisasi") < 0).sum().alias("realisasi_negatif"),
    ])
    .sort("jumlah_baris", descending=True)
    .collect()
)

negative_by_account.head(15)

akun,jumlah_baris,anggaran_negatif,realisasi_negatif
str,u32,u32,u32
"""Pembiayaan Daerah""",4215,3289,1654
"""Pengeluaran Pembiayaan""",2332,2164,2201
"""Penyertaan Modal""",2116,1906,1860
"""Pembayaran Pokok Utang""",1229,956,1093
"""Pemberian Pinjaman Daerah""",379,297,282
…,…,…,…
"""Belanja Tidak Terduga""",39,0,39
"""Lain-lain Pendapatan Sesuai de…",25,0,25
"""SiLPA TA Sebelumnya""",22,0,22


In [55]:
percentage_audit = df_clean.select([
    pl.col("persentase").min().alias("persentase_min"),
    pl.col("persentase").max().alias("persentase_max"),
    (pl.col("persentase") < 0).sum().alias("persentase_negatif"),
    (pl.col("persentase") > 100).sum().alias("persentase_di_atas_100"),
    pl.col("persentase").is_nan().sum().alias("persentase_nan"),
    pl.col("persentase").is_infinite().sum().alias("persentase_infinite"),
]).collect()

percentage_audit

persentase_min,persentase_max,persentase_negatif,persentase_di_atas_100,persentase_nan,persentase_infinite
f64,f64,u32,u32,u32,u32
-997.48,999.98,2357,125049,0,0


In [56]:
# Periksa konsistensi persentase dengan realisasi / anggaran * 100

percentage_check = (
    df_clean
    .filter(pl.col("anggaran") != 0)
    .with_columns(
        (
            pl.col("realisasi") / pl.col("anggaran") * 100
        ).alias("persentase_hitung")
    )
    .with_columns(
        (
            pl.col("persentase") - pl.col("persentase_hitung")
        ).abs().alias("selisih_persentase")
    )
)

# Ringkasan selisih antara persentase sumber dan hasil perhitungan
percentage_check.select([
    pl.len().alias("jumlah_anggaran_nonzero"),
    pl.col("selisih_persentase").max().alias("selisih_maksimum"),
    (pl.col("selisih_persentase") > 0.02)
        .sum()
        .alias("jumlah_selisih_di_atas_0_02"),
]).collect()

jumlah_anggaran_nonzero,selisih_maksimum,jumlah_selisih_di_atas_0_02
u32,f64,u32
855298,3.2376e9,4295


In [57]:
from pathlib import Path
import polars as pl

OUTPUT_PATH = OUTPUT_DIR / "apbd_pemda_indonesia_clean.parquet"

print("File ada:", OUTPUT_PATH.exists())

df_check = pl.read_parquet(OUTPUT_PATH)

print("Jumlah baris:", df_check.height)
print("Jumlah kolom:", df_check.width)

File ada: True
Jumlah baris: 905781
Jumlah kolom: 11
